In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import faiss
import pickle
import numpy as np
from langchain.document_loaders import PyPDFLoader
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer, AutoModel
import torch

c:\Users\Playdata\anaconda3\envs\vectordb_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os

path = "./data/자동차보험"
file_list = os.listdir(path)

print ("file_list: {}".format(file_list))

file_list: ['10_250228_business.pdf', '10_250228_businessCM.pdf', '10_250228_businessTM.pdf', '13_240906_comprehensive.pdf', '14_240906_comprehensiveTM.pdf', '15_250306_commercial.pdf', '15_250306_commercialCM.pdf', '15_250306_commercialTM.pdf', '17_230401_Two-WheelⅡ.pdf', '18-250221_Two-Wheel.pdf', '19_231115_Handle.pdf', '19_250221_Two-WheelCM.pdf', '20_250221_Two-WheelTM.pdf', '22_231115_Haru.pdf', '22_240418_Driver-Service.pdf', '23_240418_Driver-ServiceCM[0].pdf', '24_231115_Motorbike.pdf', '25_240131_PlatformDelivery.pdf', '26_231115_Crackdown.pdf', '27_231115_License.pdf', '28_231115_Driver.pdf', '29_231115_Electromobile.pdf', '30.231115_단기이륜차운전자.pdf', '34_241201_KB배달라이더이륜자동차보험.pdf', '4_250228_private.pdf', '4_250228_privateCM.pdf', '4_250228_privatePM.pdf', '4_250228_privateTM[0].pdf', '공동약관_개인용_250101[1].pdf', '공동약관_업무용_240906.pdf', '공동약관_영업용_240906.pdf', '공동약관_이륜차_230401[0].pdf']


In [6]:
model_name = "kakaocorp/kanana-nano-2.1b-embedding"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(model_name, trust_remote_code=True)

def get_embeddings(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")

    # pool_mask 생성 (입력 시퀀스 길이에 맞춰서 1로 설정)
    pool_mask = torch.ones(inputs["input_ids"].shape, dtype=torch.long)

    with torch.no_grad():
        outputs = model(**inputs, pool_mask=pool_mask)  # pool_mask 추가

    return outputs.embedding.numpy()

In [10]:
# 1️⃣ FAISS 저장 경로
faiss_index_path = "./faiss_index_1.bin"
metadata_path = "./documents_1.pkl"

# 2️⃣ PDF 문서 로드
documents = []
for file in file_list[:2]:
    loader = PyPDFLoader(path + "\\" + file)
    documents = loader.load()

# 3️⃣ 문서 전처리
for doc in documents:
    doc.page_content = doc.page_content.replace("\n", " ").replace("  ", " ")
    doc.metadata["source"] = file


# 1️⃣ 문서 스플리팅
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=128)
split_documents = []
for doc in documents:
    split_docs = splitter.split_documents([doc])  # 각 문서를 개별적으로 분할
    split_documents.extend(split_docs)
documents = split_documents  # 분할된 문서로 documents 업데이트


# 2️⃣ 스플릿된 문서로 벡터 생성
batch_size = 16
texts = [doc.page_content for doc in documents]
embeddings = []

for i in range(0, len(texts), batch_size):
    batch = texts[i : i + batch_size]
    batch_embeddings = get_embeddings(batch)  # 🔥 배치 단위로 임베딩 생성
    embeddings.extend(batch_embeddings)

# 3️⃣ FAISS 인덱스에 추가
embedding_dim = len(embeddings[0])
index = faiss.IndexFlatL2(embedding_dim)
index.add(np.array(embeddings, dtype=np.float32))

# 4️⃣ FAISS 인덱스 & 문서 저장
faiss.write_index(index, "./faiss_index_1.bin")
with open("./documents_1.pkl", "ab") as f:
    pickle.dump(documents, f)

print("✅ FAISS 인덱스 및 문서 저장 완료!")

✅ FAISS 인덱스 및 문서 저장 완료!


In [15]:
# 1️⃣ 저장된 FAISS 인덱스 로드
index = faiss.read_index("./faiss_index_1.bin")

# 2️⃣ 문서 정보 로드
with open("./documents_1.pkl", "rb") as f:
    documents = pickle.load(f)

# 3️⃣ 검색 수행
query = "차량 사항이 잘못된 경우"
# query = "의무보험 미가입에 따른 불이익을 알려줘"
query_embedding = get_embeddings([query])[0]  # 쿼리 임베딩
query_embedding = np.array([query_embedding], dtype=np.float32)

D, I = index.search(query_embedding, k=5)  # 가장 유사한 5개 검색

# 4️⃣ 검색 결과 출력
for idx in I[0]:
    print(f"🔹 문서 {idx}: {documents[idx].page_content[:200]}")

🔹 문서 109: 허용한 것이 금융서비스의 질을 높이고, 나아가 우리 나라 금융산업을 선진화시키기 위한 조치임을 명심하 고 고객정보의 교류를 토대로 고객 여러분들께 보다 편리하고 질 높은 선진금융서비스를 제공할 것을 약 속드리며, 고객 여러분의 고객정보의 보호 및 엄격한
🔹 문서 612: - 73 - 개시일부터 2년이 지난 후에 발생한 습관성 유산, 불임 및 인공수정 관련 합병증으로 인한 경우에는 보험금을 지급합니다.  5. 전쟁, 외국의 무력행사, 혁명, 내란, 사변, 폭동 용 어 풀 이 습관성 유산, 불임 및 인공수정 한국표준질병·사인분류상의 N96~N98에 해당하는 질병 을 말합니다.  용 어 풀 이 심신상실 정신병, 정신박약, 심한 
🔹 문서 328: - 156 - 의사소통이 불가한 경우     8) ‘말하는 기능에 뚜렷한 장해를 남긴 때’라 함은 아 래의 경우 중 하나 이상에 해당되는 때를 말한다.       가) 언어평가상 자음정확도가 50%미만인 경우       나) 언어평가상 표현언어지수 25 미만인 경우     9) ‘말하는 기능에 약간의 장해를 남긴 때’라 함은 아 래의 경우 중 하나 이상에 
🔹 문서 408: - 193 - 자의 보유 및 이용 기간을 말한다)  4. 개인정보를 제공받는 자 및 개인정보를 제공받는 자 의 개인정보 이용 목적 ③ 개인정보처리자가 정보주체로부터 법 제18조제2항제1호 및 제22조제4항에 따른 동의를 받거나 법 제22조제3항에 따라 선택적으로 동의할 수 있는 사항에 대한 동의를 받 으려는 때에는 정보주체가 동의 여부를 선택할 수 있다 는
🔹 문서 405: - 192 - <신설 2016.3.29. 2017. 7. 26.,2020.2.4.> ⑤ 보호위원회는 대통령령으로 정하는 전문기관으로 하여금 제4항에 따른 조사를 수행하게 할 수 있다.  <신설 2016.3.29. 2017. 7. 26.,2020.2.4.> 법규2 개인정보 보호법 시행령  [시행 2021.2.5.] [대통령령 제31429호, 2021.2.2.
